# 🛒 Blinkit / Swiggy Review Analyzer — V2

**Upgrade over V1:**
- ✅ Multi-label classification (a review can belong to multiple categories)
- ✅ Sentiment scoring with VADER
- ✅ Urgency detection (flags high-priority complaints)
- ✅ Zero-shot BERT labeling + SVM (CPU-friendly)
- ✅ distilBERT fine-tuning (better accuracy)
- ✅ Model comparison — best model wins
- ✅ Clean prediction pipeline ready for Streamlit deployment

**Dataset:** swiggy.csv (same as V1 — so you can compare results directly)

---

## 📦 Step 0 — Install Dependencies

In [ ]:
# Run this once
!pip install pandas numpy scikit-learn spacy vaderSentiment transformers torch datasets accelerate --quiet
!python -m spacy download en_core_web_sm --quiet

## 📊 Step 1 — Load & Explore Data (EDA)

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("swiggy.csv")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# Basic info
print("=== Column Info ===")
df.info()
print("\n=== Null Counts ===")
print(df.isnull().sum())
print("\n=== Rating Distribution ===")
print(df['rating'].value_counts().sort_index())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Rating distribution
df['rating'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Rating Distribution', fontsize=13)
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')

# Review length distribution
df['review_length'] = df['review_description'].fillna('').apply(len)
axes[1].hist(df['review_length'], bins=50, color='coral', edgecolor='black')
axes[1].set_title('Review Length Distribution', fontsize=13)
axes[1].set_xlabel('Characters')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f"Average review length: {df['review_length'].mean():.0f} characters")
print(f"Empty/null reviews: {df['review_description'].isnull().sum()}")

## 🧹 Step 2 — Data Cleaning

In [ ]:
# Drop columns not needed for analysis
cols_to_drop = ['appVersion', 'developer_response', 'developer_response_date',
                'App', 'review_date', 'thumbsUpCount', 'review_length']
cols_to_drop = [c for c in cols_to_drop if c in df.columns]  # only drop if exists
df = df.drop(columns=cols_to_drop)

# Drop rows with no review text
df = df.dropna(subset=['review_description'])
df = df[df['review_description'].str.strip() != '']
df = df.reset_index(drop=True)

print(f"Clean dataset shape: {df.shape}")
df.head()

In [ ]:
import re
import spacy

nlp = spacy.load("en_core_web_sm")

# Patterns
url_pattern   = re.compile(r"https?://\S+|www\.\S+")
emoji_pattern = re.compile(
    "[" u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F6FF"
        u"\U0001F1E0-\U0001F1FF"
    "]+", flags=re.UNICODE
)

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = url_pattern.sub("", text)
    text = emoji_pattern.sub("", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    doc = nlp(text)
    tokens = [
        token.lemma_ for token in doc
        if not token.is_stop and not token.is_punct and token.is_alpha
    ]
    return " ".join(tokens)

print("Cleaning text... (this may take a minute)")
df["processed_text"] = df["review_description"].apply(clean_text)
print("Done!")
df[["review_description", "processed_text"]].head()

## 😊 Step 3 — Sentiment Scoring (VADER)

> **Why VADER?** It's specifically tuned for short social/app reviews — handles slang, caps, punctuation like `!!!` naturally. No GPU needed.

Each review gets a **compound score** from -1 (very negative) to +1 (very positive).

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

def get_sentiment(text):
    scores = sia.polarity_scores(str(text))
    compound = scores['compound']
    if compound >= 0.05:
        label = 'positive'
    elif compound <= -0.05:
        label = 'negative'
    else:
        label = 'neutral'
    return compound, label

# Apply on original text (not cleaned — VADER needs raw text for punctuation/caps cues)
df[['sentiment_score', 'sentiment_label']] = df['review_description'].apply(
    lambda x: pd.Series(get_sentiment(x))
)

print("Sentiment Distribution:")
print(df['sentiment_label'].value_counts())
df[['review_description', 'sentiment_score', 'sentiment_label']].head(8)

In [ ]:
# Sentiment vs Rating — sanity check (should correlate)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sentiment_counts = df['sentiment_label'].value_counts()
axes[0].pie(sentiment_counts, labels=sentiment_counts.index, autopct='%1.1f%%',
            colors=['#2ecc71', '#e74c3c', '#95a5a6'], startangle=90)
axes[0].set_title('Overall Sentiment Distribution')

df.groupby('rating')['sentiment_score'].mean().plot(kind='bar', ax=axes[1],
                                                     color='mediumpurple', edgecolor='black')
axes[1].set_title('Avg Sentiment Score by Rating')
axes[1].set_xlabel('Star Rating')
axes[1].set_ylabel('Avg VADER Compound Score')
axes[1].axhline(0, color='red', linestyle='--', linewidth=1)

plt.tight_layout()
plt.show()

## 🚨 Step 4 — Urgency Detection

Flags reviews that need **immediate action** — refund disputes, legal threats, safety issues.

Returns a score from 0–3:
- **0** = Not urgent
- **1** = Mild concern
- **2** = High urgency
- **3** = Critical (legal/safety)

In [ ]:
# Urgency keyword tiers
URGENCY_TIER3 = [  # Critical
    "consumer forum", "consumer court", "legal action", "fir", "police",
    "lawyer", "fraud", "cheating", "scam", "poisoned", "food poisoning",
    "hospitalized", "sick", "injured"
]
URGENCY_TIER2 = [  # High
    "refund", "money deducted", "charged twice", "amount deducted",
    "not delivered", "never arrived", "order missing", "wrong address",
    "expired", "rotten", "cockroach", "insect", "hair", "stone"
]
URGENCY_TIER1 = [  # Mild
    "late", "delayed", "waiting", "no response", "support not helping",
    "pathetic", "worst", "terrible", "disgusting", "horrible"
]

def get_urgency(text, sentiment_score, rating):
    if not isinstance(text, str):
        return 0, 'low'
    t = text.lower()
    score = 0

    if any(kw in t for kw in URGENCY_TIER3):
        score = 3
    elif any(kw in t for kw in URGENCY_TIER2):
        score = 2
    elif any(kw in t for kw in URGENCY_TIER1):
        score = 1

    # Boost score if very negative sentiment + low rating
    if sentiment_score < -0.5 and rating <= 2 and score < 2:
        score = max(score, 1)

    label_map = {0: 'low', 1: 'medium', 2: 'high', 3: 'critical'}
    return score, label_map[score]

df[['urgency_score', 'urgency_label']] = df.apply(
    lambda row: pd.Series(get_urgency(row['review_description'], row['sentiment_score'], row['rating'])),
    axis=1
)

print("Urgency Distribution:")
print(df['urgency_label'].value_counts())
print("\nSample critical reviews:")
df[df['urgency_label'] == 'critical'][['review_description', 'urgency_score']].head(3)

## 🏷️ Step 5 — Multi-Label Categorization

**V1 problem:** One review → one label. But *"Order came late and item was missing"* is BOTH `delivery_issue` AND `order_issue`.

**V2 fix:** Each category gets its own binary column. A review can have multiple 1s.

In [ ]:
# Multi-label keyword maps
CATEGORY_KEYWORDS = {
    'delivery_issue':      ['late', 'delay', 'delayed', 'hour', 'minutes', 'waiting',
                            'deliver', 'delivery', 'slot', 'not arrived', 'never came'],
    'order_issue':         ['wrong', 'missing', 'item', 'cold', 'stale', 'spoiled',
                            'rotten', 'quality', 'packaging', 'expired', 'broken', 'damage'],
    'payment_issue':       ['refund', 'payment', 'charged', 'money', 'upi', 'card',
                            'wallet', 'transaction', 'deducted', 'cashback', 'amount'],
    'app_issue':           ['app', 'crash', 'bug', 'error', 'login', 'update',
                            'loading', 'server', 'freeze', 'slow', 'glitch'],
    'customer_support':    ['support', 'agent', 'customer care', 'helpline', 'response',
                            'no reply', 'ignored', 'chat', 'call', 'executive'],
    'positive_experience': ['awesome', 'great', 'good', 'amazing', 'love', 'fast',
                            'excellent', 'best', 'wonderful', 'happy', 'perfect', 'superb']
}

def assign_multilabels(row):
    text = str(row['review_description']).lower()
    proc = str(row['processed_text']).lower()
    rating = row['rating']
    labels = {}

    for category, keywords in CATEGORY_KEYWORDS.items():
        hit = any(kw in text or kw in proc for kw in keywords)

        # positive_experience needs high rating too
        if category == 'positive_experience':
            labels[category] = int(hit and rating >= 4)
        else:
            labels[category] = int(hit)

    # If nothing matched and rating is low → flag as delivery_issue (common default complaint)
    if sum(labels.values()) == 0:
        if rating <= 2:
            labels['delivery_issue'] = 1  # fallback for vague negative reviews

    return pd.Series(labels)

print("Assigning multi-labels...")
label_cols = list(CATEGORY_KEYWORDS.keys())
df[label_cols] = df.apply(assign_multilabels, axis=1)

print("\nLabel frequency (how many reviews have each label):")
print(df[label_cols].sum().sort_values(ascending=False))
print(f"\nReviews with 2+ labels (true multi-label): {(df[label_cols].sum(axis=1) >= 2).sum()}")

In [ ]:
# Visualize label distribution
label_counts = df[label_cols].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 4))
label_counts.plot(kind='bar', color='teal', edgecolor='black')
plt.title('Multi-Label Category Distribution', fontsize=13)
plt.xlabel('Category')
plt.ylabel('Number of Reviews')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 🤖 Step 6A — Approach 1: Zero-Shot BERT Labeling + SVM Classifier

**Idea:** Use `facebook/bart-large-mnli` (zero-shot) to generate better labels → train a fast SVM on TF-IDF features.

**Pro:** No fine-tuning needed, SVM inference is instant.

> ⚠️ Zero-shot labeling is slow on CPU. Run on a sample or use GPU for full dataset.

In [ ]:
from transformers import pipeline

# Load zero-shot classifier (downloads ~1.5GB first time)
print("Loading zero-shot classifier...")
zero_shot = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=-1  # CPU; change to 0 if you have GPU
)
print("Loaded!")

CANDIDATE_LABELS = [
    "delivery delay", "wrong or missing item",
    "payment or refund problem", "app technical issue",
    "poor customer support", "positive experience"
]

LABEL_MAP = {
    "delivery delay": "delivery_issue",
    "wrong or missing item": "order_issue",
    "payment or refund problem": "payment_issue",
    "app technical issue": "app_issue",
    "poor customer support": "customer_support",
    "positive experience": "positive_experience"
}

In [ ]:
# Run on a sample (500 reviews) to keep it fast on CPU
# Increase sample_size or use full df if you have GPU
sample_size = min(500, len(df))
df_sample = df.sample(n=sample_size, random_state=42).copy()

def get_zeroshot_label(text):
    text = str(text)[:512]  # Truncate to BART's limit
    result = zero_shot(text, CANDIDATE_LABELS)
    top_label = result['labels'][0]
    return LABEL_MAP[top_label]

print(f"Running zero-shot on {sample_size} reviews... (grab a coffee ☕)")
df_sample['zs_label'] = df_sample['review_description'].apply(get_zeroshot_label)
print("Done!")
print(df_sample['zs_label'].value_counts())

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# Train SVM on zero-shot labels
X_zs = df_sample['processed_text']
y_zs = df_sample['zs_label']

X_train_zs, X_test_zs, y_train_zs, y_test_zs = train_test_split(
    X_zs, y_zs, test_size=0.2, random_state=42, stratify=y_zs
)

tfidf_zs = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.9)
X_train_vec = tfidf_zs.fit_transform(X_train_zs)
X_test_vec  = tfidf_zs.transform(X_test_zs)

svm_zs = LinearSVC(class_weight='balanced', max_iter=2000)
svm_zs.fit(X_train_vec, y_train_zs)

y_pred_zs = svm_zs.predict(X_test_vec)
acc_zs = accuracy_score(y_test_zs, y_pred_zs)

print(f"\n=== Approach 1: Zero-Shot BERT Labels + SVM ===")
print(f"Accuracy: {acc_zs:.4f}")
print("\nClassification Report:")
print(classification_report(y_test_zs, y_pred_zs))

## 🤖 Step 6B — Approach 2: distilBERT Fine-Tuning

**Idea:** Fine-tune `distilbert-base-uncased` on our rule-labeled data. Smaller & faster than full BERT, still much better than TF-IDF.

> ⚠️ Fine-tuning works best with GPU. On CPU it's slow — reduce `num_train_epochs` to 1 for a quick test.

In [ ]:
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)
from datasets import Dataset
import torch

# Use the primary single label from rule-based categorization for fine-tuning
# We pick the first matched label as the primary label
def get_primary_label(row):
    for col in label_cols:
        if row[col] == 1:
            return col
    return 'delivery_issue'  # fallback

df['primary_label'] = df.apply(get_primary_label, axis=1)

# Encode labels
label2id = {l: i for i, l in enumerate(label_cols)}
id2label = {i: l for l, i in label2id.items()}
df['label_id'] = df['primary_label'].map(label2id)

print("Label encoding:")
print(label2id)
print("\nLabel distribution:")
print(df['primary_label'].value_counts())

In [ ]:
# Prepare dataset — use a manageable sample for CPU
bert_sample_size = min(2000, len(df))
df_bert = df.sample(n=bert_sample_size, random_state=42).reset_index(drop=True)

train_df, test_df = train_test_split(df_bert, test_size=0.2, random_state=42,
                                      stratify=df_bert['label_id'])

# Tokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize_function(examples):
    return tokenizer(
        examples['review_description'],
        truncation=True,
        padding='max_length',
        max_length=128
    )

train_dataset = Dataset.from_pandas(train_df[['review_description', 'label_id']])
test_dataset  = Dataset.from_pandas(test_df[['review_description', 'label_id']])

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset  = test_dataset.map(tokenize_function, batched=True)

# Rename label column
train_dataset = train_dataset.rename_column('label_id', 'labels')
test_dataset  = test_dataset.rename_column('label_id', 'labels')
train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

print(f"Train size: {len(train_dataset)} | Test size: {len(test_dataset)}")

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score as sk_accuracy

# Load model
model_bert = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=len(label_cols),
    id2label=id2label,
    label2id=label2id
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {'accuracy': sk_accuracy(labels, predictions)}

training_args = TrainingArguments(
    output_dir='./distilbert_reviews',
    num_train_epochs=3,           # Reduce to 1 for quick CPU test
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    report_to='none'  # Disable wandb/tensorboard
)

trainer = Trainer(
    model=model_bert,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

print("Starting distilBERT fine-tuning...")
trainer.train()
print("Fine-tuning complete!")

In [ ]:
# Evaluate distilBERT
results_bert = trainer.evaluate()
acc_bert = results_bert['eval_accuracy']

print(f"\n=== Approach 2: distilBERT Fine-Tuned ===")
print(f"Accuracy: {acc_bert:.4f}")

# Save model
trainer.save_model('./distilbert_reviews/best_model')
tokenizer.save_pretrained('./distilbert_reviews/best_model')
print("\nModel saved to ./distilbert_reviews/best_model")

## 🏆 Step 7 — Model Comparison & Final Decision

In [ ]:
results = {
    'V1 Baseline (TF-IDF + SVM)':    0.0,   # Fill in from V1 notebook
    'V2 Zero-Shot BERT + SVM':        acc_zs,
    'V2 distilBERT Fine-Tuned':       acc_bert
}

print("=" * 45)
print(f"{'Model':<30} {'Accuracy':>10}")
print("=" * 45)
for model_name, acc in results.items():
    print(f"{model_name:<30} {acc:>10.4f}")
print("=" * 45)

best_model = max(results, key=results.get)
print(f"\n✅ Winner: {best_model}")

# Plot comparison
plt.figure(figsize=(9, 4))
colors = ['#95a5a6', '#3498db', '#e74c3c']
bars = plt.bar(results.keys(), results.values(), color=colors, edgecolor='black')
plt.ylim(0, 1)
plt.title('Model Accuracy Comparison — V1 vs V2', fontsize=13)
plt.ylabel('Accuracy')
plt.xticks(rotation=15, ha='right')
for bar, val in zip(bars, results.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 🔧 Step 8 — Final Unified Prediction Pipeline

This is what the Streamlit app will use.

In [ ]:
import pickle

# Save SVM + TF-IDF for Streamlit (fast inference)
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_zs, f)
with open('svm_classifier.pkl', 'wb') as f:
    pickle.dump(svm_zs, f)

print("Saved: tfidf_vectorizer.pkl, svm_classifier.pkl")
print("Saved: ./distilbert_reviews/best_model/ (use for higher accuracy)")

In [ ]:
# -------------------------------------------------------
# UNIFIED PREDICTION FUNCTION
# Input : raw review text + star rating
# Output: category, sentiment, urgency
# -------------------------------------------------------

def analyze_review(text, rating=3, use_bert=False):
    """
    Full pipeline:
    - Cleans text
    - Predicts category (SVM or BERT)
    - Scores sentiment (VADER)
    - Detects urgency

    Returns a dict with all outputs.
    """
    # 1. Clean
    cleaned = clean_text(text)

    # 2. Category prediction
    if use_bert:
        from transformers import pipeline as hf_pipeline
        bert_pipe = hf_pipeline(
            'text-classification',
            model='./distilbert_reviews/best_model',
            device=-1
        )
        category = bert_pipe(text[:512])[0]['label']
    else:
        vec = tfidf_zs.transform([cleaned])
        category = svm_zs.predict(vec)[0]

    # 3. Sentiment
    sentiment_score, sentiment_label = get_sentiment(text)

    # 4. Urgency
    urgency_score, urgency_label = get_urgency(text, sentiment_score, rating)

    return {
        'input_text':      text,
        'category':        category,
        'sentiment_score': round(sentiment_score, 3),
        'sentiment_label': sentiment_label,
        'urgency_score':   urgency_score,
        'urgency_label':   urgency_label
    }

# -------------------------------------------------------
# Test it!
# -------------------------------------------------------
test_reviews = [
    ("My order is 45 minutes late, this is ridiculous.", 1),
    ("Received rotten tomatoes and bread was missing.", 1),
    ("Money got deducted but order was never placed. Where is my refund?", 1),
    ("The app keeps crashing whenever I try to checkout.", 2),
    ("Super fast delivery! Got groceries in 10 minutes. Amazing!", 5),
    ("I will take this to consumer forum. This is fraud!", 1),
    ("Customer support is pathetic, nobody responds.", 2),
]

print(f"{'Review':<55} {'Category':<22} {'Sentiment':<12} {'Urgency'}")
print("-" * 110)
for review, rating in test_reviews:
    result = analyze_review(review, rating, use_bert=False)
    print(f"{review[:52]:<55} {result['category']:<22} "
          f"{result['sentiment_label']:<12} {result['urgency_label']}")

## ✅ Summary — V1 vs V2

| Feature | V1 | V2 |
|---|---|---|
| Classification | Single-label (5 categories) | Multi-label (6 categories) |
| Labeling strategy | Keyword rules only | Zero-shot BERT + rules |
| Model | LinearSVC | Zero-shot SVM + distilBERT |
| Sentiment | ❌ | ✅ VADER (compound score + label) |
| Urgency detection | ❌ | ✅ 4-tier scoring |
| Deployment ready | ❌ | ✅ Saved models + unified pipeline |

**Next:** `app.py` — Streamlit dashboard for single review + bulk CSV analysis.